<a href="https://colab.research.google.com/github/illelias/CLIP/blob/main/CLIP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch pillow

In [1]:
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel
import itertools

# Load model
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Load image
image = Image.open("/content/me_dog.jpg").convert("RGB")

# -------------------------
# DEFINE COMPONENTS
# -------------------------

genders = ["man", "woman"]

races = ["white", "black", "asian", "latino"]

jobs = ["civilian", "soldier", "police officer", "firefighter", "businessman"]

animals = [
    "",
    "standing with a dog",
    "standing with a corgi"
]

trees = [
    "",
    "under a tree",
    "under a cherry blossom tree",
    "under a palm tree",
    "under an oak tree"
]

# -------------------------
# GENERATE DESCRIPTIONS
# -------------------------

descriptions = []

for gender, race, job, animal, tree in itertools.product(
    genders, races, jobs, animals, trees
):
    sentence = f"a photo of a {race} {job} {gender}"

    if animal:
        sentence += f" {animal}"
    if tree:
        sentence += f" {tree}"

    descriptions.append(sentence)

# Remove duplicates just in case
descriptions = list(set(descriptions))

print(f"Total descriptions: {len(descriptions)}")
print(descriptions[:10])  # preview

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Total descriptions: 720
['a photo of a white soldier man standing with a corgi', 'a photo of a black firefighter man standing with a dog under an oak tree', 'a photo of a black veteran woman under an oak tree', 'a photo of a white veteran man under an oak tree', 'a photo of a white soldier woman under an oak tree', 'a photo of a asian veteran woman standing with a corgi under a palm tree', 'a photo of a black soldier woman standing with a corgi under a tree', 'a photo of a asian businessman man standing with a dog under a palm tree', 'a photo of a asian firefighter man standing with a corgi under a cherry blossom tree', 'a photo of a latino police officer woman']


In [2]:
inputs = processor(
    text=descriptions,
    images=image,
    return_tensors="pt",
    padding=True
)

with torch.no_grad():
    outputs = model(**inputs)

logits_per_image = outputs.logits_per_image
probs = logits_per_image.softmax(dim=1)

# Get top 5 matches
topk = torch.topk(probs, k=5)

print("\nTop 5 matches:\n")
for idx, score in zip(topk.indices[0], topk.values[0]):
    print(f"{descriptions[idx]} --> {score.item():.4f}")


Top 5 matches:

a photo of a latino civilian man standing with a corgi under a cherry blossom tree --> 0.1506
a photo of a latino civilian man standing with a dog under a cherry blossom tree --> 0.0851
a photo of a asian civilian man standing with a corgi under a cherry blossom tree --> 0.0806
a photo of a latino veteran man standing with a corgi under a cherry blossom tree --> 0.0790
a photo of a white civilian man standing with a corgi under a cherry blossom tree --> 0.0708
